<center><b>Python Programming for Multilingual Text</b></center>
<center>3-2: Historical Named Entity Recognition with Stanza</center>

---

# Named Entity Recognition with Stanza

## Overview

First we started with one letter and then we scaled our text analysis tools to the whole Montagu corpus, and along the way we ran into POS tagging and lemmatization. This notebook introduces a related but different task: **Named Entity Recognition (NER)**.

Where POS tagging asks "what part of speech is this word?", NER asks "does this word refer to a person, a place, an organization, a date, or something else in the world?"

We will follow the same pattern as before: start with one letter, understand the tool deeply, then scale up to the full corpus. New to this notebook:

1. **Setup**: installing and loading Stanza
2. **NER on a single sentence**: understanding entity types and the output format
3. **NER on Letter 35**: applying it to a full letter
4. **A key limitation**: why a model trained on modern news text struggles with 18th-century letters
5. **Scaling to the corpus**: running NER on all letters and storing results
6. **Aggregating entities**: counting people and places across the corpus

---

**Dataset:** Lady Mary Wortley Montagu's *Turkish Embassy Letters* (1716–1718)

---


## Step 1: Setup

[Stanza](https://stanfordnlp.github.io/stanza/ner.html) is a different library from NLTK, so it has its own installation and download steps. Unlike NLTK, where we downloaded small individual resources (`punkt`, `stopwords`, etc.), Stanza downloads a full **pipeline** which is a bundle of neural network models that work together (tokenizer, POS tagger, NER tagger, and so on).

[PyTorch](https://pypi.org/project/torch/) is a machine learning framework for Python used primarily for deep learning applications like computer vision and natural language processing. It is one of the [most important deep learning frameworks](https://pytorch.org/) today.



In [ ]:
import torch
# Stanza's official models need weights_only=False on torch 2.2 
_torch_load = torch.load
torch.load = lambda *args, **kwargs: _torch_load(*args, **{**kwargs, "weights_only": False})


import stanza
import pandas as pd
from collections import Counter
from itertools import chain

stanza.download('en') # download English model

/Users/mervetekgurler/anaconda3/envs/cts2000/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-04 09:12:43 INFO: Downloaded file to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/resources.json
2026-11-04 09:12:43 INFO: Downloading default packages for language: en (English) ...
2026-11-04 09:12:44 INFO: File exists: /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/en/default.zip
2026-11-04 09:12:49 INFO: Finished downloading models and saved to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources


[['zip', 'default.zip']]

In [3]:
# Let's test the example from the repository itself
# https://github.com/stanfordnlp/stanza
# This allows us to test that the code is working as intended.

nlp = stanza.Pipeline('en') # This sets up a default neural pipeline in English
doc = nlp("Barack Obama was born in Hawaii. He was elected president in 2008.")
doc.sentences[0].print_dependencies()

2026-11-04 09:12:55 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-11-04 09:12:56 INFO: Downloaded file to /Users/mervetekgurler/Library/Caches/stanza/1.15.0/resources/resources.json
2026-11-04 09:12:57 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined_nocharlm         |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-11-04 09:12:57 INFO: Using device: cpu
2026-11-04 09:12:57 INFO: Loading: tokenize
2026-11-04 09:12:59 INFO: Loading: mwt
2026-11-04 0

('Barack', 4, 'nsubj:pass')
('Obama', 1, 'flat')
('was', 4, 'aux:pass')
('born', 0, 'root')
('in', 6, 'case')
('Hawaii', 4, 'obl')
('.', 4, 'punct')


> **What just happened?** `stanza.Pipeline(...)` loaded the models into memory and built a function-like object, `nlp`, that we can now call on any piece of text. This loading step is the slow part — once `nlp` exists, processing text with it is much faster. This is why we build the pipeline once, outside of any loop, rather than rebuilding it for every letter.

In [ ]:
# processors='tokenize,ner' loads only what we need:
# - tokenize: splits text into words and sentences (NER depends on this)
# - ner: the named entity recognizer itself

# We could also run this instead of all the stanza models to only load the tokenizer, which we need to be able to run the NER and the NER model

ner = stanza.Pipeline(lang='en', processors='tokenize,ner')

---
## Step 2: NER on a single sentence

Before we turn to a the letters, let's pick a sentence from one letter and run NER on it

In [4]:
# it is a sentence from letter 32
sample = "My side-saddle is the first that was ever seen in this part of the world, and is gazed at with as much wonder as the ship of Columbus in the first discovery of America."

doc = nlp(sample)

for sentence in doc.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}") #!r:25 makes print statement look neat like a table

'first'                   ORDINAL
'Columbus'                GPE
'first'                   ORDINAL
'America'                 GPE


Let's unpack this:

`doc = nlp(sample)` -> running our pipeline on a string returns a `Document` object, not a plain string or list. This is different from NLTK, where most functions returned lists or tuples directly.

`doc.sentences` -> a `Document` is broken into `Sentence` objects, even if your input is one sentence. This matters for the corpus-level work later, since Stanza's own sentence splitting can disagree with NLTK's `sent_tokenize()`.

`sentence.ents` -> each `Sentence` has an `.ents` attribute: a list of entities Stanza found in it.

`ent.text` and `ent.type` -> each entity has the original text span (`ent.text`) and a category label (`ent.type`), such as `PERSON`, `GPE` (geo-political entity — countries, cities, states), `ORG`, or `DATE`.

In [10]:
# this print statement from stanza documentation shows us each word in the sentence and how it was/was not tagged

print(*[f'token: {token.text}\tner: {token.ner}' for sent in doc.sentences for token in sent.tokens], sep='\n')

token: My	ner: O
token: side	ner: O
token: -	ner: O
token: saddle	ner: O
token: is	ner: O
token: the	ner: O
token: first	ner: S-ORDINAL
token: that	ner: O
token: was	ner: O
token: ever	ner: O
token: seen	ner: O
token: in	ner: O
token: this	ner: O
token: part	ner: O
token: of	ner: O
token: the	ner: O
token: world	ner: O
token: ,	ner: O
token: and	ner: O
token: is	ner: O
token: gazed	ner: O
token: at	ner: O
token: with	ner: O
token: as	ner: O
token: much	ner: O
token: wonder	ner: O
token: as	ner: O
token: the	ner: O
token: ship	ner: O
token: of	ner: O
token: Columbus	ner: S-GPE
token: in	ner: O
token: the	ner: O
token: first	ner: S-ORDINAL
token: discovery	ner: O
token: of	ner: O
token: America	ner: S-GPE
token: .	ner: O


### Entity types you'll commonly see

| Type | Meaning | Example |
|---|---|---|
| `PERSON` | People, including fictional | *Obama, Montagu* |
| `GPE` | Countries, cities, states | *Hawaii, Constantinople* |
| `LOC` | Non-GPE locations: mountains, bodies of water, regions | *the Bosphorus* |
| `ORG` | Companies, agencies, institutions | *the Senate* |
| `DATE` | Absolute or relative dates | *1717, last Tuesday* |
| `NORP` | Nationalities, religious or political groups | *French, Catholic* |
| `FAC` | Buildings, airports, bridges | *the Globe Theatre* |


> Stanza's English NER model is trained on the **[OntoNotes](https://catalog.ldc.upenn.edu/LDC2013T19)** dataset, a large collection of modern news, talk shows, web text, and similar sources. They have 18 categories for entities, which you can read more about [here on page 21](https://catalog.ldc.upenn.edu/docs/LDC2013T19/OntoNotes-Release-5.0.pdf)

---
## Step 3: NER on Letter 35

Let's bring back Letter 35, the same letter we worked with closely in 2-4 and again at the start of 3-1.

In [5]:
# Load the corpus
montagu = pd.read_csv('../../data/montagu/montagu_letters_v3.csv')
montagu = montagu[['filename', 'title', 'body']]
montagu.head()


,filename,title,body
0,letter_1.txt,﻿LETTER 1.,"I FLATTER, myself, dear sister, that I shall g..."
1,letter_2.txt,LET. II,"I MAKE haste to tell you, dear Madam, that, af..."
2,letter_3.txt,LET. III,"I AM extremely sorry, my dear S. that your fea..."
3,letter_4.txt,LET. IV.,IF my lady ---- could have any notion of the ...
4,letter_5.txt,LET. V.,"AFTER five days travelling post, I could not s..."


In [11]:
# eding the \n line breaks from the body for stanza
montagu['body'] = montagu['body'].apply(lambda text: " ".join(text.split()))

In [12]:
letter_35 = montagu['body'].iloc[34]
print(letter_35[:300])

I HAVE had the advantage of very fine weather, all my journey; and as the summer is now in its beauty, I enjoyed the pleasure of fine prospects; and the meadows being full of all sorts of garden flowers, and sweet herbs, my berlin perfumed the air as it pressed them. The grand signior furnished us w


In [13]:
# Run the Stanza pipeline on the full letter
doc_35 = nlp(letter_35)

In [14]:
print(f"Number of sentences Stanza found: {len(doc_35.sentences)}")

Number of sentences Stanza found: 71


> Compare this sentence count to the one we got from NLTK's `sent_tokenize()` (68 sentences for Letter 35). Are they the same? If not, why might two different tools disagree about where sentences begin and end?

In [15]:
# Extract all entities from doc_35 and print them as (text, type)

letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]

for text, etype in letter_35_entities:
    print(f"{text!r:30} {etype}")

'thirty'                       CARDINAL
'five'                         CARDINAL
'Asia'                         LOC
'Ciorlei'                      GPE
'Turkish'                      NORP
'One'                          CARDINAL
'English'                      LANGUAGE
'Marmora'                      LOC
'Propontis'                    LOC
'Selivrea'                     GPE
'thirty-two'                   CARDINAL
'Greek'                        NORP
'one'                          CARDINAL
'Greek'                        NORP
'Roman'                        NORP
'Mary'                         PERSON
'St Luke'                      PERSON
'Madona'                       PERSON
'Italy'                        GPE
'Greeks'                       NORP
'Bujuk Cekmege'                FAC
'Great Bridge'                 FAC
'Kujuk Cekmege'                FAC
'Little Bridge'                FAC
'two'                          CARDINAL
'every night'                  TIME
'fifty'                        CARDINAL


Let's unpack the list comprehension:

```python
letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]
```

This is a **nested** list comprehension — two `for` clauses instead of one. Read it left to right, but think about it inside-out:

`for sentence in doc_35.sentences` -> outer loop, over each sentence in the document

`for ent in sentence.ents` -> inner loop, over each entity within that sentence

`(ent.text, ent.type)` -> what we keep from each entity: the text span and its label

This is equivalent to:

```python
letter_35_entities = []
for sentence in doc_35.sentences:
    for ent in sentence.ents:
        letter_35_entities.append((ent.text, ent.type))
```

Both versions do the same thing. The nested comprehension is more compact once you're comfortable with it, but there's no shame in writing the longer version first and condensing it later.

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Look closely at the output above. Are there any entities that are clearly wrong — mislabeled, or not entities at all? Are there any real people or places in the letter that Stanza missed entirely?
</div>

---
## Step 4: A key limitation — why this matters for historical text

Stanza's English NER model was trained on OntoNotes: modern news wire, broadcast conversation, web text, and similar genres from the late 20th and early 21st century. Montagu was writing private letters in 1716–1718, describing a part of the world with person and place names with several competing English spellings that does not appear in the training data.

Researchers who work on **historical NER** as a subfield have documented this gap extensively. The challenges they describe consistently include archaic spelling and inconsistent orthography, named entities that no longer exist or have changed meaning, and a basic shortage of historical text to train on compared to the enormous amount of modern text available online. None of this is a flaw in Stanza specifically — it's a structural feature of how all modern NER models get trained, and it generalizes to spaCy, FLAIR, or any other off-the-shelf system you might reach for.

> This is not a reason to abandon NER for historical text. It's a reason to treat its output as a **starting point for inquiry**, not a finished answer. We need to read the output critically to ensure that we are actually we are actually capturing what is valuable and interesting to us.

In [16]:
# Let's look at a sentence built specifically to surface this gap.
# Try swapping in a sentence or two of your own choosing from Letter 35.

test_sentence = "We lay that night at a town called Bujuk Cekmege, or Great Bridge; and the night following, at Kujuk Cekmege, or Little Bridge; in a very pleasant lodging, formerly a monastery of dervises; having before it a large court, encompassed with marble cloisters, with a good fountain in the middle."

doc_test = nlp(test_sentence)
for sentence in doc_test.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

'Bujuk Cekmege'           FAC
'Great Bridge'            FAC
'Kujuk Cekmege'           FAC
'Little Bridge'           FAC


---
## Step 5: Scaling to the corpus

Now that we have looked closely at one letter, let's apply the same pipeline to all of Montagu's letters.

> ⚠️ Running a neural pipeline on every letter in the corpus is much slower than the simple string operations. Expect this cell to take noticeably longer to run. This is normal — Stanza is doing far more computation per word than `.split()` or even NLTK's `word_tokenize()`.

In [17]:
# Run NER on every letter and store the resulting Document object
montagu['ner_doc'] = montagu['body'].apply(lambda text: nlp(text))

It took ca. 24 minutes to run!

We've stored the full `Document` object for each letter in ner_doc, not just the entities. This mirrors what we did with `pos_tags`: keep the intermediate result around so you don't have to recompute it if you want to extract something different from it later.

Now let's pull just the entities out into their own column.

In [18]:
def extract_entities(doc):
    """Extract all (text, type) entity pairs from a Stanza Document."""
    return [(ent.text, ent.type) for sentence in doc.sentences for ent in sentence.ents]

montagu['entities'] = montagu['ner_doc'].apply(extract_entities)
montagu[['title', 'entities']].head()

,title,entities
0,﻿LETTER 1.,"[(two days, DATE), (Rotterdam, GPE), (Helvoets..."
1,LET. II,"[(Madam, PERSON), (Holland, GPE), (twenty, CAR..."
2,LET. III,"[(S., PERSON), (Nottingham, GPE), (two, CARDIN..."
3,LET. IV.,"[(two last days, DATE), (Nimeguen, GPE), (Rein..."
4,LET. V.,"[(five days, DATE), (Germany, GPE), (Cologn, G..."


In [19]:
montagu.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   filename  52 non-null     str   
 1   title     52 non-null     str   
 2   body      52 non-null     str   
 3   ner_doc   52 non-null     object
 4   entities  52 non-null     object
dtypes: object(2), str(3)
memory usage: 2.2+ KB


> Side note: We actually ran the full nlp pipeline on all letters and not the ner only pipeline. (I forgot I switched the variable names). This is not a bad thing, we just did more work and we might as well save it better for the future.

In [20]:
# Let's extract our stanza annotations
montagu['sentences']   = montagu['ner_doc'].apply(lambda doc: [s.text for s in doc.sentences])
montagu['num_sents']   = montagu['sentences'].apply(len)
montagu['words']       = montagu['ner_doc'].apply(lambda doc: [w.text for s in doc.sentences for w in s.words])
montagu['lemmas']      = montagu['ner_doc'].apply(lambda doc: [w.lemma for s in doc.sentences for w in s.words])
montagu['pos_tags']    = montagu['ner_doc'].apply(lambda doc: [(w.text, w.upos) for s in doc.sentences for w in s.words])
montagu['xpos_tags']   = montagu['ner_doc'].apply(lambda doc: [(w.text, w.xpos) for s in doc.sentences for w in s.words])  # Penn Treebank tags, like NLTK's
montagu['morph_feats'] = montagu['ner_doc'].apply(lambda doc: [(w.text, w.feats) for s in doc.sentences for w in s.words])  # tense, number, etc.

# Dependencies: (word, relation, head word)
montagu['dependencies'] = montagu['ner_doc'].apply(
    lambda doc: [(w.text, w.deprel, s.words[w.head - 1].text if w.head > 0 else 'ROOT')
                 for s in doc.sentences for w in s.words])

# Sentiment per sentence: 0 = negative, 1 = neutral, 2 = positive
montagu['sentiment']      = montagu['ner_doc'].apply(lambda doc: [s.sentiment for s in doc.sentences])
montagu['mean_sentiment'] = montagu['sentiment'].apply(lambda x: sum(x) / len(x))

# Constituency parse trees, saved as strings
montagu['constituency'] = montagu['ner_doc'].apply(lambda doc: [str(s.constituency) for s in doc.sentences])

In [21]:
montagu.head()

,filename,title,body,ner_doc,entities,sentences,num_sents,words,lemmas,pos_tags,xpos_tags,morph_feats,dependencies,sentiment,mean_sentiment,constituency
0,letter_1.txt,﻿LETTER 1.,"I FLATTER, myself, dear sister, that I shall g...","[\n [\n {\n ""id"": 1,\n ""text"": ""...","[(two days, DATE), (Rotterdam, GPE), (Helvoets...","[I FLATTER, myself, dear sister, that I shall ...",18,"[I, FLATTER, ,, myself, ,, dear, sister, ,, th...","[I, flatter, ,, myself, ,, dear, sister, ,, th...","[(I, PRON), (FLATTER, VERB), (,, PUNCT), (myse...","[(I, PRP), (FLATTER, VBP), (,, ,), (myself, PR...","[(I, Case=Nom|Number=Sing|Person=1|PronType=Pr...","[(I, nsubj, FLATTER), (FLATTER, root, ROOT), (...","[1, 2, 1, 1, 2, 0, 1, 1, 1, 2, 1, 0, 2, 2, 2, ...",1.222222,"[(ROOT (S (NP (PRP I)) (VP (VBP FLATTER) (, ,)..."
1,letter_2.txt,LET. II,"I MAKE haste to tell you, dear Madam, that, af...","[\n [\n {\n ""id"": 1,\n ""text"": ""...","[(Madam, PERSON), (Holland, GPE), (twenty, CAR...","[I MAKE haste to tell you, dear Madam, that, a...",12,"[I, MAKE, haste, to, tell, you, ,, dear, Madam...","[I, make, haste, to, tell, you, ,, dear, Madam...","[(I, PRON), (MAKE, VERB), (haste, NOUN), (to, ...","[(I, PRP), (MAKE, VBP), (haste, NN), (to, TO),...","[(I, Case=Nom|Number=Sing|Person=1|PronType=Pr...","[(I, nsubj, MAKE), (MAKE, root, ROOT), (haste,...","[2, 1, 2, 2, 2, 2, 2, 1, 1, 0, 1, 2]",1.500000,[(ROOT (S (NP (PRP I)) (VP (VBP MAKE) (NP (NN ...
2,letter_3.txt,LET. III,"I AM extremely sorry, my dear S. that your fea...","[\n [\n {\n ""id"": 1,\n ""text"": ""...","[(S., PERSON), (Nottingham, GPE), (two, CARDIN...","[I AM extremely sorry, my dear S. that your fe...",18,"[I, AM, extremely, sorry, ,, my, dear, S., tha...","[I, be, extremely, sorry, ,, my, dear, S., tha...","[(I, PRON), (AM, AUX), (extremely, ADV), (sorr...","[(I, PRP), (AM, VBP), (extremely, RB), (sorry,...","[(I, Case=Nom|Number=Sing|Person=1|PronType=Pr...","[(I, nsubj, sorry), (AM, cop, sorry), (extreme...","[1, 1, 1, 1, 1, 2, 0, 2, 2, 2, 1, 1, 1, 1, 2, ...",1.166667,[(ROOT (S (S (NP (PRP I)) (VP (VBP AM) (ADJP (...
3,letter_4.txt,LET. IV.,IF my lady ---- could have any notion of the f...,"[\n [\n {\n ""id"": 1,\n ""text"": ""...","[(two last days, DATE), (Nimeguen, GPE), (Rein...",[IF my lady ---- could have any notion of the ...,12,"[IF, my, lady, ----, could, have, any, notion,...","[if, my, lady, ----, could, have, any, notion,...","[(IF, SCONJ), (my, PRON), (lady, NOUN), (----,...","[(IF, IN), (my, PRP$), (lady, NN), (----, ,), ...","[(IF, None), (my, Case=Gen|Number=Sing|Person=...","[(IF, mark, have), (my, nmod:poss, lady), (lad...","[0, 1, 0, 0, 2, 0, 1, 2, 1, 1, 1, 1]",0.833333,[(ROOT (S (SBAR (IN IF) (S (NP (PRP$ my) (NN l...
4,letter_5.txt,LET. V.,"AFTER five days travelling post, I could not s...","[\n [\n {\n ""id"": 1,\n ""text"": ""...","[(five days, DATE), (Germany, GPE), (Cologn, G...","[AFTER five days travelling post, I could not ...",19,"[AFTER, five, days, travelling, post, ,, I, co...","[after, five, day, travel, post, ,, I, could, ...","[(AFTER, ADP), (five, NUM), (days, NOUN), (tra...","[(AFTER, IN), (five, CD), (days, NNS), (travel...","[(AFTER, None), (five, NumForm=Word|NumType=Ca...","[(AFTER, case, post), (five, nummod, days), (d...","[0, 0, 1, 1, 1, 2, 0, 0, 1, 2, 0, 1, 1, 1, 1, ...",0.789474,[(ROOT (S (PP (IN AFTER) (S (NP (CD five) (NNS...


In [22]:
# doc.to_dict() turns each Document into plain lists/dicts (tokens, lemmas, POS, deps, NER tags, etc)
# sentiment and constituency trees are not included, so we grab those separately
montagu['doc_dict'] = montagu['ner_doc'].apply(lambda doc: doc.to_dict())

# pickle keeps Python objects intact (CSV would turn lists into strings)
montagu.drop(columns=['ner_doc']).to_pickle('../../data/montagu/montagu_stanza_full.pkl')


In [ ]:
# We could read this in later with the Stanza document

# from stanza.models.common.doc import Document

# montagu = pd.read_pickle('../../data/montagu/montagu_stanza_full.pkl')
# montagu['ner_doc'] = montagu.apply(lambda row: Document(row['doc_dict'], text=row['body']), axis=1)

In [23]:
# Save a smaller CSV to explore later. We leave out the Stanza objects and the huge nested columns.

montagu_to_save = montagu.drop(columns=['ner_doc', 'doc_dict', 'constituency'])
montagu_to_save.to_csv('../../data/montagu/montagu_letters_v4.csv', index=False)
print("Saved! Columns now:", montagu_to_save.columns.tolist())


Saved! Columns now: ['filename', 'title', 'body', 'entities', 'sentences', 'num_sents', 'words', 'lemmas', 'pos_tags', 'xpos_tags', 'morph_feats', 'dependencies', 'sentiment', 'mean_sentiment']


In [ ]:
# and here is how we would access our csv later

# montagu = pd.read_csv("../../data/montagu/montagu_letters_v4.csv")

# montagu.head()

---
## Step 6: Aggregating entities across the corpus

With entities extracted for every letter, we can ask corpus-level questions: who does Montagu mention most? What places recur across her letters?

### People

In [24]:
# Flatten all entities into one list, same pattern as chain.from_iterable in 3-1
all_entities = list(chain.from_iterable(montagu['entities']))

# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

Top 20 most-mentioned people:
  W: 16
  Turk: 8
  Adieu: 7
  Homer: 6
  Fatima: 6
  I'll: 4
  Mustapha: 4
  Paul Rycaut: 4
  Constantinople: 4
  Sancta Sophia: 4
  Madam: 3
  S.: 3
  Amelia: 3
  Virgil: 3
  Helen: 3
  Constantine: 3
  Lucretia: 3
  Bartholomew: 2
  Lorrain: 2
  Amphitrion: 2


In [25]:
ent_list = montagu["entities"]
ent_list[:5]

0    [(two days, DATE), (Rotterdam, GPE), (Helvoets...
1    [(Madam, PERSON), (Holland, GPE), (twenty, CAR...
2    [(S., PERSON), (Nottingham, GPE), (two, CARDIN...
3    [(two last days, DATE), (Nimeguen, GPE), (Rein...
4    [(five days, DATE), (Germany, GPE), (Cologn, G...
Name: entities, dtype: object

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Scan this list. Are these all genuinely distinct people, or are they even all people?
</div>

### Places

In [ ]:
### YOUR CODE HERE
# Following the pattern above, extract GPE and LOC entities and count the most common ones.


In [26]:
# This is how I would do it

places = [text for text, etype in all_entities if etype in ('GPE', 'LOC')]
place_counts = Counter(places)

print("Top 20 most-mentioned places:")
for place, count in place_counts.most_common(20):
    print(f"  {place}: {count}")

Top 20 most-mentioned places:
  Vienna: 33
  England: 26
  London: 25
  Constantinople: 22
  Paris: 16
  Hungary: 14
  Germany: 13
  Danube: 11
  Belgrade: 11
  France: 8
  Europe: 7
  Peterwaradin: 7
  Italy: 7
  Dresden: 6
  Asia: 6
  Turkey: 6
  Hanover: 5
  Pera: 5
  Carthage: 5
  Genoa: 4


In [27]:
# Pick one place from your list above and confirm it actually appears where you'd expect
search_term = "Vienna"  # change this to check on a new term

matches = montagu[montagu['body'].str.contains(search_term, case=False, na=False)]
print(f"'{search_term}' appears in {len(matches)} letter(s):")
print(matches['title'].tolist())

'Vienna' appears in 15 letter(s):
['LET. VII.', 'LET. IX.', 'LET. X.', 'LET. XI.', 'LET. XII.', 'LET.  XIV.', 'LET. XV.', 'LET. XVII.', 'LET. XVIII.', 'LET. XIX.', 'LET. XX.', 'LET. XXI.', 'LET. XXIII.', 'LET. XXXIII.', 'LET. XXXVII.']


## Can we map these places?

to be continued!